In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import xarray as xr
import matplotlib.pyplot as plt
import argparse
import datetime as dt

from utils.misc_functions import *
from utils.YParams import *

In [ ]:
def build_model_input_from_netcdf(nc_file, p, include_metar=True):
    # Match ADAF inference.py behavior: open with netcdf4 engine
    ds = xr.open_dataset(nc_file, engine="netcdf4")

    H = p.img_size_y
    W = p.img_size_x

    lon = np.array(ds.coords['lon'].values)[:W]
    lat = np.array(ds.coords['lat'].values)[:H]
    topo = np.array(ds[['z']].to_array())[:, :H, :W]

    inp_hrrr = np.array(ds[p.inp_hrrr_vars].to_array())[:, :H, :W]
    inp_hrrr = np.squeeze(inp_hrrr)

    obs = np.array(ds[p.inp_obs_vars].to_array())[:, -p.obs_time_window:, :H, :W]
    
    # --- METAR Filtering Block ---
    if not include_metar:
        # Extract the spatial 2D grid matching your target image size
        obs_source = np.array(ds['obs_source'].values)[:H, :W]
        
        # Wherever obs_source is 2 (METAR), zero out all variables and time windows
        # obs shape is (variables, time_window, H, W)
        obs[:, :, obs_source == 2] = 0
    
    obs_tar = obs[:, -1]
    obs_tar_mask = (obs_tar != 0).astype(np.float32)

    # Training-style obs input construction
    if p.hold_out_obs:
        obs_flat = obs[0, 0].flatten()
        obs_idx = np.where(obs_flat != 0)[0]
        hold_out_num = int(len(obs_idx) * p.hold_out_obs_ratio)

        if p.obs_mask_seed != 0:
            np.random.seed(p.obs_mask_seed)

        np.random.shuffle(obs_idx)
        hold_out_idx = obs_idx[:hold_out_num]

        obs_mask = np.zeros(obs_flat.shape, dtype=np.float32)
        obs_mask[hold_out_idx] = 1.0
        obs_mask = obs_mask.reshape(obs[0, 0].shape[0], obs[0, 0].shape[1])

        inp_obs = obs * (1 - obs_mask)
        inp_obs = inp_obs.reshape((-1, H, W))
    else:
        inp_obs = obs.reshape((-1, H, W))

    field_tar = np.array(ds[p.field_tar_vars].to_array())[:, :H, :W]

    field_obs_tar = field_tar.copy()
    field_obs_tar[obs_tar_mask == 1] = 0
    field_obs_tar += obs_tar

    if p.learn_residual:
        field_tar = field_tar - inp_hrrr
        obs_tar = obs_tar - inp_hrrr
        field_obs_tar = field_obs_tar - inp_hrrr

    inp = np.concatenate((inp_hrrr, inp_obs, topo), axis=0).astype(np.float32)

    aux = {
        'lat': lat,
        'lon': lon,
        'inp_hrrr': inp_hrrr.astype(np.float32),
        'inp_obs': inp_obs.astype(np.float32),
        'topo': topo.astype(np.float32),
        'target_field': field_tar.astype(np.float32),
        'target_obs': obs_tar.astype(np.float32),
        'target_field_obs': field_obs_tar.astype(np.float32),
        'obs_tar_mask': obs_tar_mask.astype(np.float32),
    }
    ds.close()
    return inp, aux

#####################################################

def load_checkpoint_weights(model, checkpoint_file, map_device):
    ckpt = torch.load(checkpoint_file, map_location=map_device)

    if 'model_state' in ckpt:
        state = ckpt['model_state']
    elif 'state_dict' in ckpt:
        state = ckpt['state_dict']
    else:
        raise KeyError("Checkpoint does not contain 'model_state' or 'state_dict'.")

    # Remove DDP 'module.' prefix if present
    clean_state = {}
    for k, v in state.items():
        clean_key = k[7:] if k.startswith('module.') else k
        clean_state[clean_key] = v

    missing, unexpected = model.load_state_dict(clean_state, strict=False)
    return ckpt, missing, unexpected

#####################################################

def reverse_norm_from_stats(data, variable_names, stats_file):
    """Reverse min-max normalization from [-1, 1] back to physical units.

    This is intentionally separate from model inference so it can be disabled
    or adjusted without touching the inference path.
    """
    stats = pd.read_csv(stats_file)
    stats_map = stats.set_index("variable")

    vmin = np.array([stats_map.loc[v, "min"] for v in variable_names], dtype=np.float32)
    vmax = np.array([stats_map.loc[v, "max"] for v in variable_names], dtype=np.float32)

    arr = np.asarray(data, dtype=np.float32)

    # Expected channel-first arrays: [C,H,W] or [C,T,H,W]
    if arr.ndim >= 3:
        if arr.shape[0] != len(variable_names):
            raise ValueError(
                f"Channel mismatch: data has {arr.shape[0]} channels, "
                f"but variable_names has {len(variable_names)} entries."
            )
        reshape = (len(variable_names),) + (1,) * (arr.ndim - 1)
        vmin_b = vmin.reshape(reshape)
        vmax_b = vmax.reshape(reshape)
        return (arr + 1.0) * (vmax_b - vmin_b) / 2.0 + vmin_b

    # Optional scalar/2D support for single-variable arrays
    if arr.ndim == 2 and len(variable_names) == 1:
        return (arr + 1.0) * (vmax[0] - vmin[0]) / 2.0 + vmin[0]

    raise ValueError("Unsupported array shape for reverse normalization.")

#####################################################

def plot_output_channel(results_dict, channel_name, 
                        channel_to_select="prediction_analysis_unnorm", 
                        colorbar_scale_style="normal",
                        abs_min=None, abs_max=None,
                        title_str=None, 
                        colorbar_label=None, 
                        plot_savepath=None, 
                        cmap='bwr'):
    """Plot one unnormalized output channel by variable name (e.g., 'output_t')."""
    output_names = results_dict['output_channel_names']
    if channel_name not in output_names:
        raise KeyError(
            f"Unknown channel '{channel_name}'. Available: {output_names}"
        )

    idx = output_names.index(channel_name)
    arr = results_dict[channel_to_select][idx]

    # Use georeferenced axes from saved lat/lon arrays
    lat = np.asarray(results_dict['lat'])
    lon = np.asarray(results_dict['lon'])
    extent = [float(np.min(lon)), float(np.max(lon)), float(np.min(lat)), float(np.max(lat))]

    # Determine colorbar limits based on scale style
    if colorbar_scale_style == "centered":
        abs_max = np.nanmax(np.abs(arr))
        vmin, vmax = -abs_max, abs_max
    elif colorbar_scale_style == 'extreme':
        vmin, vmax = abs_min, abs_max #user-defined limits
    elif colorbar_scale_style == "normal":
        vmin, vmax = None, None
    else:
        raise ValueError("colorbar_scale_style must be either 'normal' or 'extreme' or 'centered'")

    fig, ax = plt.subplots(figsize=(12, 6))  
    
    # Pass vmin and vmax to imshow
    im = ax.imshow(arr, origin='lower', cmap=cmap, extent=extent, aspect='auto', vmin=vmin, vmax=vmax)

    if title_str is None:
        ax.set_title(f"{channel_name} (unnormalized) | min={np.nanmin(arr):.3f}, max={np.nanmax(arr):.3f}")
    else:
        ax.set_title(title_str)
    
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

    # Tie colorbar to axis so colorbar height matches plot height
    cbar = fig.colorbar(im, ax=ax, pad=0.02, fraction=0.016)
    if colorbar_label is None:
        cbar.set_label(f"{channel_name} value (unnormalized)")
    else:
        cbar.set_label(colorbar_label)
        
    if plot_savepath is not None:
        plt.savefig(plot_savepath, dpi=300, bbox_inches='tight')
    
    plt.tight_layout()
    plt.show()

#####################################################

def run_model_inference(model, nc_path, params, stats_path, device, include_metar=True):
    """
    Runs inference on a single NetCDF file, unnormalizes the outputs, 
    and packages the arrays and metadata into a structured dictionary.
    """
    # 1. Load data and push to device
    inp_np, aux = build_model_input_from_netcdf(nc_path, params, include_metar=include_metar)
    inp_tensor = torch.from_numpy(inp_np).unsqueeze(0).to(device)

    print("Input tensor shape:", tuple(inp_tensor.shape))
    print("Expected in_chans:", params.in_chans)

    # 2. Model Inference
    model.eval() # Good practice to ensure model is in evaluation mode
    with torch.no_grad():
        pred_tensor = model(inp_tensor)

    pred_np = pred_tensor.squeeze(0).detach().cpu().numpy()
    print("Prediction tensor shape:", tuple(pred_tensor.shape))
    print("Prediction array shape:", pred_np.shape)

    # 3. Extract normalized outputs
    pred_residual_norm = pred_np.copy()
    target_residual_norm = aux['target_field'].copy()

    # 4. Unnormalize residual outputs using RTMA variable stats
    pred_residual_unnorm = reverse_norm_from_stats(pred_residual_norm, params.field_tar_vars, stats_path)
    target_residual_unnorm = reverse_norm_from_stats(target_residual_norm, params.field_tar_vars, stats_path)

    # Unnormalize HRRR background for residual reconstruction
    hrrr_unnorm = reverse_norm_from_stats(aux['inp_hrrr'].copy(), params.inp_hrrr_vars, stats_path)

    # 5. Reconstruct full analysis fields if model learns residuals
    if params.learn_residual:
        pred_analysis_unnorm = pred_residual_unnorm + hrrr_unnorm
        target_analysis_unnorm = target_residual_unnorm + hrrr_unnorm
    else:
        pred_analysis_unnorm = pred_residual_unnorm
        target_analysis_unnorm = target_residual_unnorm

    print("Unnormalization complete.")
    print("pred_residual_unnorm shape:", pred_residual_unnorm.shape)
    print("pred_analysis_unnorm shape:", pred_analysis_unnorm.shape)

    # 6. Channel mappings
    output_channel_names = [f"output_{v.split('rtma_', 1)[1]}" if v.startswith('rtma_') else f"output_{v}" for v in params.field_tar_vars]

    channel_maps = {
        'input_hrrr': {i: v for i, v in enumerate(params.inp_hrrr_vars)},
        'input_obs': {i: v for i, v in enumerate(params.inp_obs_vars)},
        'output': {i: v for i, v in enumerate(output_channel_names)},
        'target_field': {i: v for i, v in enumerate(params.field_tar_vars)},
    }

    # 7. Package results
    results = {
        # Normalized outputs (original)
        'prediction_tensor': pred_tensor,
        'prediction_array_norm': pred_np,
        'input_tensor': inp_tensor,
        'input_array_norm': inp_np,
        'target_field_array_norm': aux['target_field'],
        'target_obs_array_norm': aux['target_obs'],
        'target_field_obs_array_norm': aux['target_field_obs'],

        # Unnormalized outputs (new)
        'prediction_residual_unnorm': pred_residual_unnorm,
        'target_residual_unnorm': target_residual_unnorm,
        'hrrr_unnorm': hrrr_unnorm,
        'prediction_analysis_unnorm': pred_analysis_unnorm,
        'target_analysis_unnorm': target_analysis_unnorm,

        # Channel metadata
        'channel_maps': channel_maps,
        'output_channel_names': output_channel_names,

        # Masks/coords
        'obs_tar_mask_array': aux['obs_tar_mask'],
        'lat': aux['lat'],
        'lon': aux['lon'],
    }

    return results

In [ ]:
#########################################################

In [ ]:
model_number = 16233268
model_name = "ckpt" #"best_ckpt"

base_dir = "/scratch3/BMC/wrfruc/aschein/ADAF_RTMA"
data_dir = "/scratch5/BMC/ai-datadepot/projects/aschein/ADAF_new/data"
ckpt_path = os.path.join(base_dir, f"training_runs/{model_number}/{model_name}.tar")
stats_path = os.path.join(base_dir, "data_preparation/stats.csv")

# Ensure model package is importable from notebook
if base_dir not in os.sys.path:
    os.sys.path.append(base_dir)

from models.encdec import EncDec

config_filepath="./config/params_default.yaml"
params = YParams(config_filepath)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
# print("NetCDF:", nc_path)
print("Checkpoint:", ckpt_path)
print("Stats:", stats_path)

In [ ]:
# Load model
model = EncDec(params).to(device)
ckpt, missing, unexpected = load_checkpoint_weights(model, ckpt_path, device)
model.eval()

print("Checkpoint keys:", list(ckpt.keys()))
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

In [ ]:
#########################################################

In [ ]:
# Change date as desired
analysis_time = dt.datetime(2022,10,1,6)
nc_path = os.path.join(data_dir, f"valid_data/{analysis_time.strftime('%Y-%m-%d_%H.nc')}") 
params["hold_out_obs"] = False #Don't hold out any obs

In [ ]:
%%time
## 1. Model given all obs
params["hold_out_obs"] = False #Don't hold out any obs

results_all_obs = run_model_inference(model, nc_path, params, stats_path, device, include_metar=True)

In [ ]:
%%time
## 2. Model given only Mesonet
results_mesonet_only = run_model_inference(model, nc_path, params, stats_path, device, include_metar=False)

In [ ]:
%%time
## 3. Model given no obs
params["hold_out_obs"] = True
params["hold_out_obs_ratio"] = 1 #Deny everything

results_no_obs = run_model_inference(model, nc_path, params, stats_path, device, include_metar=False)

In [ ]:
## 4. Differences

unnorm_keys = [
    'prediction_residual_unnorm', 
    'target_residual_unnorm', 
    'hrrr_unnorm', 
    'prediction_analysis_unnorm', 
    'target_analysis_unnorm'
]

results_all_obs_minus_mesonet_only = results_all_obs.copy()
results_all_obs_minus_no_obs = results_all_obs.copy()

for key in unnorm_keys:
    results_all_obs_minus_mesonet_only[key] = results_all_obs[key] - results_mesonet_only[key]
    results_all_obs_minus_no_obs[key] = results_all_obs[key] - results_no_obs[key]

# Compare all obs output to RTMA
results_all_obs_minus_rtma = results_all_obs.copy()
results_all_obs_minus_rtma['prediction_analysis_unnorm'] = results_all_obs['prediction_analysis_unnorm'] - results_all_obs['target_analysis_unnorm']

In [ ]:
#########################################################

In [ ]:
units_dict={'output_t':'C',
            'output_q':'g/kg',
            'output_u10':'m/s',
            'output_v10':'m/s'}

plot_dir = f"/scratch3/BMC/wrfruc/aschein/ADAF_RTMA/Plots"

channel = "output_v10"

In [ ]:
plot_output_channel(results_all_obs, 
                    f"{channel}", 
                    channel_to_select='prediction_analysis_unnorm',
                    title_str=f"{channel}, Mesonet+METAR obs, {analysis_time.strftime('%Y-%m-%d %H')} UTC", 
                    colorbar_label=f"Value ({units_dict[channel]})", 
                    plot_savepath=None)#f"{plot_dir}/{channel}_model_{model_number}_{model_name}.png") 

In [ ]:
plot_output_channel(results_all_obs, 
                    f"{channel}", 
                    title_str=f"{channel}, model innovation, all obs, {analysis_time.strftime('%Y-%m-%d %H')} UTC",
                    channel_to_select="prediction_residual_unnorm",
                    colorbar_scale_style='extreme',
                    abs_min=-5, abs_max=10,
                    colorbar_label=f"Value ({units_dict[channel]})", 
                    plot_savepath=None)#f"{plot_dir}/innovation_{channel}_model_{model_number}_{model_name}.png") 

In [ ]:
plot_output_channel(results_all_obs_minus_mesonet_only, 
                    f"{channel}", 
                    title_str=f"{channel} comparison, Mesonet+METAR minus Mesonet only, {analysis_time.strftime('%Y-%m-%d %H')} UTC",
                    colorbar_scale_style='centered',
                    colorbar_label=f"Difference ({units_dict[channel]})", 
                    plot_savepath=None)#f"{plot_dir}/difference_{channel}_model_{model_number}_{model_name}.png") 

In [ ]:
plot_output_channel(results_all_obs_minus_no_obs, 
                    f"{channel}", 
                    title_str=f"{channel} comparison, Mesonet+METAR minus no obs, {analysis_time.strftime('%Y-%m-%d %H')} UTC", 
                    colorbar_scale_style='centered',
                    colorbar_label=f"Difference ({units_dict[channel]})", 
                    plot_savepath=None)#f"{plot_dir}/difference_noobs_{channel}_model_{model_number}_{model_name}.png") 

In [ ]:
plot_output_channel(results_all_obs_minus_rtma, 
                    f"{channel}", 
                    title_str=f"{channel}, model error (output minus RTMA), all obs, {analysis_time.strftime('%Y-%m-%d %H')} UTC",
                    channel_to_select="prediction_analysis_unnorm",
                    colorbar_scale_style='extreme',
                    abs_min=-10, abs_max=10,
                    colorbar_label=f"Value ({units_dict[channel]})", 
                    plot_savepath=None)#f"{plot_dir}/error_{channel}_model_{model_number}_{model_name}.png") 